# Reproducibility & Experiment Tracking

Training a model is only half the job. Six months from now you (or a teammate) will need to answer: *which hyperparameters produced that 0.94-AUC model, on what data, with what code?* If you can't answer that, the result isn't science, it's a lucky accident.

**Experiment tracking** makes every run reproducible and comparable by recording four things per run:

- **Params** &mdash; the knobs you turned (`n_estimators`, `max_depth`, learning rate, ...).
- **Metrics** &mdash; the numbers you care about (`accuracy`, `roc_auc`, loss, ...).
- **Artifacts** &mdash; the files a run produced (the serialized model, plots, a confusion matrix, ...).
- **Environment / versions** &mdash; the random **seed**, the **data** version, the **code** (git) version, and package versions.

Tools like **MLflow** and **Weights & Biases (W&B)** capture all of this for you and add a UI to compare runs. Below we use MLflow *if it is installed*, and otherwise fall back to a tiny local tracker we build by hand &mdash; so the notebook runs fully offline while teaching the exact same concepts.

## Installing a real tracker (optional)

MLflow and W&B are normally installed with pip:

```bash
pip install mlflow    # open-source, self-hosted; `mlflow ui` opens a local dashboard at :5000
pip install wandb     # Weights & Biases: hosted dashboards, great for team / cloud runs
```

Neither is installed in this environment (and we can't reach the network), so **everything below is written to work without them**. Every `mlflow` call is guarded by `try/except ImportError`, with a hand-rolled local tracker as the fallback. The fallback mirrors MLflow's core concepts: a *run* context that logs **params + metrics** to per-run JSON files and copies **artifacts** into `./mlruns_lite/<run_id>/`, plus a function that loads every run into a pandas **leaderboard**.

In [ ]:
import numpy as np                 # arrays + the RNG we seed for reproducibility
import pandas as pd                # the run leaderboard lives in a DataFrame
import matplotlib.pyplot as plt    # plotting metric-vs-hyperparameter across runs
import json                        # params/metrics are logged as small JSON files
import shutil                      # copy artifacts + reset the tracking dir between runs
import time                        # timestamp component of each run id
import uuid                        # random component of each run id (avoids collisions)
from pathlib import Path           # tidy, OS-independent path handling
import joblib                      # serialize / reload the trained sklearn model artifact

from sklearn.datasets import make_classification            # small synthetic dataset (offline)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split        # train/test split
from sklearn.metrics import accuracy_score, roc_auc_score   # the two metrics we log

# --- Reproducibility knob #1: a single global seed reused EVERYWHERE ---
# Every model, split, and RNG below is fed this same SEED so that re-running the whole
# notebook reproduces identical numbers. Change it in ONE place and the whole run shifts.
SEED = 42
np.random.seed(SEED)

# --- Try MLflow; degrade gracefully to our hand-rolled tracker if it's missing ---
# This is the pattern that lets the SAME code run on a laptop with MLflow installed and
# in a locked-down/offline box without it. MLFLOW is the flag every later cell branches on.
try:
    import mlflow                 # noqa: F401  (only imported to test availability here)
    MLFLOW = True
except ImportError:
    mlflow = None                 # keep the name defined so references don't NameError
    MLFLOW = False

print("mlflow available:", MLFLOW, "-> using", "MLflow" if MLFLOW else "local JSON tracker")

## 1. Data

We train on a small synthetic classification problem from `make_classification` &mdash; no network, no files, and fully reproducible via `random_state=SEED`. The dataset is our **data version**: because it's generated deterministically from a seed, logging that seed *is* logging the data.

We hold out 25% for testing and `stratify=y` to keep the class balance identical across the split.

In [ ]:
# Deterministic dataset: same SEED -> byte-identical X, y on every run (data reproducibility).
X, y = make_classification(
    n_samples=1500,      # small enough to keep the whole sweep well under ~30s
    n_features=20,       # 20 features...
    n_informative=8,     # ...of which 8 actually carry signal
    n_redundant=4,       # 4 are linear combos of the informative ones (adds mild difficulty)
    class_sep=1.0,       # moderate separation so metrics land in an interesting range
    random_state=SEED,   # <-- makes the "data version" reproducible
)

# 75/25 split. stratify=y preserves the 0/1 ratio; random_state pins WHICH rows go where.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print(f"train: {X_train.shape}, test: {X_test.shape}, positives in train: {y_train.mean():.2f}")

## 2. A tiny local experiment tracker (the offline fallback)

MLflow's mental model is simple, so we can reproduce its essentials in ~40 lines:

| MLflow concept | What it does | Our local equivalent |
|---|---|---|
| `mlflow.start_run()` | opens a run, returns a handle | `start_local_run()` context manager |
| `mlflow.log_param(k, v)` | records a hyperparameter | `run.log_param(k, v)` -> `params.json` |
| `mlflow.log_metric(k, v)` | records a metric | `run.log_metric(k, v)` -> `metrics.json` |
| `mlflow.log_artifact(path)` | copies a file into the run | `run.log_artifact(path)` -> copy into run dir |
| `mlflow ui` | dashboard to compare runs | `load_leaderboard()` -> a pandas DataFrame |

Each run gets its own folder `./mlruns_lite/<run_id>/` holding `params.json`, `metrics.json`, and any artifacts. A context manager guarantees the JSON is flushed to disk even if training raises.

In [ ]:
from contextlib import contextmanager

# Root folder that plays the role of MLflow's "tracking store".
TRACKING_DIR = Path("./mlruns_lite")


class LocalRun:
    '''Handle returned inside a `with start_local_run() as run:` block.

    Mirrors the small slice of the MLflow API we actually use: log_param / log_metric /
    log_artifact. Params and metrics are buffered in dicts and flushed to JSON on exit.
    '''

    def __init__(self, run_dir):
        self.run_dir = run_dir     # this run's private folder: ./mlruns_lite/<run_id>/
        self.params = {}           # buffered hyperparameters -> params.json
        self.metrics = {}          # buffered metrics         -> metrics.json

    def log_param(self, key, value):
        # Store one hyperparameter (e.g. n_estimators=200). Mirrors mlflow.log_param.
        self.params[key] = value

    def log_metric(self, key, value):
        # Store one metric as a float (JSON has no numpy types). Mirrors mlflow.log_metric.
        self.metrics[key] = float(value)

    def log_artifact(self, path):
        # Copy a produced file (here: the serialized model) INTO the run folder, exactly
        # like mlflow.log_artifact snapshots a file into the run's artifact store.
        shutil.copy(path, self.run_dir / Path(path).name)

    def _flush(self):
        # Persist the buffered dicts to disk as human-readable JSON (the "database").
        (self.run_dir / "params.json").write_text(json.dumps(self.params, indent=2))
        (self.run_dir / "metrics.json").write_text(json.dumps(self.metrics, indent=2))


@contextmanager
def start_local_run():
    '''Open a local run: make its folder, yield a LocalRun, flush JSON on exit.

    The try/finally is the whole point of a context manager here: even if training blows
    up inside the `with` block, whatever was logged so far is still written to disk.
    '''
    # Sortable, unique run id: a timestamp (human-readable, chronological) + short random hex.
    run_id = time.strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
    run_dir = TRACKING_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)   # create ./mlruns_lite/<run_id>/
    run = LocalRun(run_dir)
    try:
        yield run                                # hand control back to the `with` body
    finally:
        run._flush()                             # ALWAYS persist params + metrics
    # Return value isn't yielded, but callers read run.run_dir.name as the run id.


print("Local tracker ready. Runs will be written under:", TRACKING_DIR.resolve())

## 3. `train_and_log(params)` &mdash; the core of an experiment

One function, one experiment. It:

1. **Sets the seed** (reproducibility knob #1, re-applied per run so run order can't change results).
2. **Builds and trains** a model from the given hyperparameters (RandomForest or GradientBoosting).
3. **Computes metrics** &mdash; `accuracy` and `roc_auc` on the held-out test set.
4. **Logs** params + metrics + the serialized model **artifact** &mdash; via MLflow if available, else via our local tracker.

Both logging branches expose the *same* API (`log_param` / `log_metric` / `log_artifact`), so the training body doesn't care which backend it's talking to.

In [ ]:
def build_model(params):
    '''Factory: turn a plain params dict into a configured (untrained) sklearn model.'''
    if params["model"] == "rf":
        # RandomForest: bag of decision trees, averaged. Seeded for reproducibility.
        return RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=SEED,
        )
    elif params["model"] == "gb":
        # GradientBoosting: trees fit sequentially on residuals; learning_rate scales each.
        return GradientBoostingClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params.get("learning_rate", 0.1),
            random_state=SEED,
        )
    raise ValueError(f"unknown model type: {params['model']!r}")


def train_and_log(params):
    '''Train one model with `params`, evaluate it, and log everything. Returns (run_id, metrics).'''
    # (1) Reproducibility: re-seed at the START of every run so results are independent of
    #     how many runs ran before this one (global RNG state can't leak between runs).
    np.random.seed(SEED)

    # (2) Build + fit the model on the training split.
    model = build_model(params)
    model.fit(X_train, y_train)

    # (3) Metrics on the HELD-OUT test set. roc_auc needs probabilities, not hard labels.
    proba = model.predict_proba(X_test)[:, 1]      # P(class = 1) for each test row
    preds = model.predict(X_test)                  # hard 0/1 predictions
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "roc_auc":  roc_auc_score(y_test, proba),
    }

    # (4) Log params + metrics + the serialized model artifact.
    #     Same three-call recipe regardless of backend -> we just branch on MLFLOW.
    if MLFLOW:
        # ---- MLflow backend (used automatically if mlflow is installed) ----
        with mlflow.start_run() as active:
            for k, v in params.items():
                mlflow.log_param(k, v)                       # log each hyperparameter
            for k, v in metrics.items():
                mlflow.log_metric(k, v)                      # log each metric
            tmp = Path("_model_tmp.joblib")
            joblib.dump(model, tmp)                          # serialize model to a temp file...
            mlflow.log_artifact(str(tmp))                    # ...then snapshot it into the run
            tmp.unlink()                                     # clean up the temp file
            run_id = active.info.run_id
    else:
        # ---- Hand-rolled local backend (the offline fallback) ----
        with start_local_run() as run:
            for k, v in params.items():
                run.log_param(k, v)                          # -> params.json
            for k, v in metrics.items():
                run.log_metric(k, v)                         # -> metrics.json
            tmp = Path("_model_tmp.joblib")
            joblib.dump(model, tmp)                          # serialize model to a temp file...
            run.log_artifact(tmp)                            # ...then COPY it into the run dir
            tmp.unlink()                                     # clean up the temp file
            run_id = run.run_dir.name

    return run_id, metrics


# Smoke-test the function on one config before running the full sweep.
_rid, _m = train_and_log({"model": "rf", "n_estimators": 100, "max_depth": 5})
print(f"logged run {_rid}  ->  accuracy={_m['accuracy']:.3f}  roc_auc={_m['roc_auc']:.3f}")

## 4. Run a small hyperparameter sweep

Real experiment tracking earns its keep across *many* runs. We reset the tracking directory (so re-running the notebook doesn't pile up stale runs) and then sweep a handful of configs across both model families. Each call to `train_and_log` becomes one logged, comparable run.

In [ ]:
# Reset the tracking store so the leaderboard reflects ONLY this notebook execution.
# (In a real project you would NEVER delete history -- here it keeps re-runs deterministic.)
if not MLFLOW and TRACKING_DIR.exists():
    shutil.rmtree(TRACKING_DIR)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

# The sweep: a few RandomForest and GradientBoosting configs of varying capacity.
param_grid = [
    {"model": "rf", "n_estimators":  50, "max_depth": 3},
    {"model": "rf", "n_estimators": 100, "max_depth": 5},
    {"model": "rf", "n_estimators": 200, "max_depth": 8},
    {"model": "rf", "n_estimators": 300, "max_depth": None},   # None = grow trees fully
    {"model": "gb", "n_estimators": 100, "max_depth": 3, "learning_rate": 0.10},
    {"model": "gb", "n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
]

# Run every config; each iteration trains a model and logs one run.
for i, params in enumerate(param_grid, 1):
    rid, m = train_and_log(params)
    print(f"[{i}/{len(param_grid)}] {params}")
    print(f"        run_id={rid}  accuracy={m['accuracy']:.3f}  roc_auc={m['roc_auc']:.3f}")

## 5. Load every run into a leaderboard

This is the offline stand-in for `mlflow ui`. We walk `./mlruns_lite/`, read each run's JSON, and assemble a `DataFrame` &mdash; one row per run, columns for params and metrics. Sorting by `roc_auc` gives the leaderboard and the best run.

(If MLflow were installed, the same table comes from `mlflow.search_runs()`.)

In [ ]:
def load_leaderboard(tracking_dir=TRACKING_DIR):
    '''Read every run's params.json + metrics.json into one tidy DataFrame.'''
    if MLFLOW:
        # With MLflow, the tracking server already indexes runs -> one call returns them all.
        return mlflow.search_runs()

    rows = []
    for run_dir in sorted(tracking_dir.glob("*")):        # each subfolder is one run
        if not run_dir.is_dir():
            continue
        params = json.loads((run_dir / "params.json").read_text())    # the knobs
        metrics = json.loads((run_dir / "metrics.json").read_text())  # the results
        # Flatten params + metrics into a single row, tagged with the run id.
        rows.append({"run_id": run_dir.name, **params, **metrics})
    return pd.DataFrame(rows)


# Build the leaderboard and sort best-first by roc_auc.
leaderboard = load_leaderboard()
leaderboard = leaderboard.sort_values("roc_auc", ascending=False).reset_index(drop=True)

print("=== RUN LEADERBOARD (sorted by roc_auc) ===")
print(leaderboard.to_string(index=False))

# The best run is simply the top row after sorting.
best = leaderboard.iloc[0]
print(f"\nBest run: {best['run_id']}  ->  roc_auc={best['roc_auc']:.4f}  accuracy={best['accuracy']:.4f}")

## 6. Plot a metric against a hyperparameter

A picture of *how* a knob moves the metric is often more useful than the raw table. Here: `roc_auc` vs `n_estimators` for the RandomForest runs.

In [ ]:
# Keep only the RandomForest runs and order them by tree count for a clean line.
rf_runs = leaderboard[leaderboard["model"] == "rf"].sort_values("n_estimators")

plt.figure(figsize=(7, 4))
# Line + markers: roc_auc as a function of the n_estimators hyperparameter.
plt.plot(rf_runs["n_estimators"], rf_runs["roc_auc"], "o-", color="#cc6666", label="RandomForest")
# Annotate each point with its max_depth so both swept knobs are visible at once.
for _, r in rf_runs.iterrows():
    plt.annotate(f"depth={r['max_depth']}", (r["n_estimators"], r["roc_auc"]),
                 textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.title("roc_auc vs n_estimators (RandomForest sweep)")
plt.xlabel("n_estimators"); plt.ylabel("roc_auc"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Reload the best run's model and reproduce its score

The payoff of logging the **artifact**: we can load the exact serialized model from the best run and re-score it on the test set. Because the data is seed-fixed and the model was frozen at log time, the reproduced `roc_auc` must match the leaderboard value *exactly* &mdash; that equality is what "reproducible" means.

In [ ]:
# Locate the best run's saved model artifact...
if MLFLOW:
    # MLflow stores artifacts under the run; download_artifacts returns a local path.
    artifact_dir = mlflow.artifacts.download_artifacts(run_id=best["run_id"])
    model_path = Path(artifact_dir) / "_model_tmp.joblib"
else:
    # Local tracker: the artifact was copied straight into ./mlruns_lite/<run_id>/.
    model_path = TRACKING_DIR / best["run_id"] / "_model_tmp.joblib"

# ...reload it and re-score on the SAME held-out test set.
reloaded = joblib.load(model_path)
reproduced_auc = roc_auc_score(y_test, reloaded.predict_proba(X_test)[:, 1])

print(f"logged  roc_auc: {best['roc_auc']:.6f}")
print(f"reloaded roc_auc: {reproduced_auc:.6f}")
# A hard assert turns "looks the same" into a guarantee -- reproducibility, verified.
assert abs(reproduced_auc - best["roc_auc"]) < 1e-9, "reloaded model did NOT reproduce the score!"
print("\nReproduced the best run's score exactly from its logged artifact.")

## 8. MLflow vs W&B, and a reproducibility checklist

**What MLflow tracks**
- **Params** (`log_param`), **metrics** (`log_metric`, with step-wise history for curves), **artifacts** (`log_artifact` &mdash; any file), and **models** (`mlflow.sklearn.log_model`, which also records the input signature and environment).
- All of it is browsable in a UI: run `mlflow ui` and open <http://localhost:5000> to sort, filter, and compare runs side by side &mdash; exactly what our `load_leaderboard` table imitates offline.

**How W&B compares**
- Same primitives (`wandb.init`, `wandb.config` for params, `wandb.log` for metrics, `wandb.Artifact` for files), but the dashboard is **hosted** by default &mdash; excellent for team sharing, live training curves, sweeps, and cloud runs. MLflow is trivially **self-hosted / offline**, which is why it's the friendlier default for a local or air-gapped setup.
- Both let you swap backends without touching your training loop &mdash; precisely the `try/except` pattern used in this notebook.

**Reproducibility checklist** (log these every run so any result can be regenerated):
1. **Seeds** &mdash; one global seed, re-applied per run (`np.random.seed`, framework seeds, `random_state`).
2. **Data version** &mdash; a hash, path, or generator seed pinning the exact dataset (here: `make_classification(random_state=SEED)`).
3. **Code version** &mdash; the git commit SHA (MLflow auto-logs `mlflow.source.git.commit`).
4. **Environment** &mdash; package versions (`pip freeze` / `conda env export` / `requirements.txt`, or MLflow's logged `conda.yaml`).
5. **Params & metrics** &mdash; every hyperparameter in, every metric out.
6. **Artifacts** &mdash; the serialized model (and plots), so you can reload and re-score without retraining &mdash; as demonstrated above.

Tick all six and any run in your history becomes a button you can press to get the same number back.